# Audit Logs — Dev Log

## Objetivo e papel no pipeline

O módulo `core/audit_logs` implementa o **log de auditoria imutável** do AthenaGov AI: um registro append-only, encadeado por hash (hash-chain estilo blockchain simplificado), de tudo que acontece nos módulos de governança.

**Ideia central:** todo módulo do sistema — PII Detection, Policy Engine, Prompt Security, Trust Score, RIPD Engine, Regulatory RAG — deveria, ao concluir uma operação relevante (um scan, uma decisão de política, um cálculo de score, uma geração de RIPD, uma consulta ao RAG), registrar um evento aqui via `default_logger().record_event(...)`.

Isso dá ao Governance Copilot (orquestrador) e a um auditor humano uma trilha **verificável e à prova de adulteração silenciosa**: se alguém editar um evento passado sem recalcular a cadeia inteira, `verify_chain()` detecta a quebra.

Os tipos `AuditEvent` e `AuditEventType` vêm de `shared/schemas.py` — este módulo não redefine nenhum contrato, apenas implementa a lógica de encadeamento/persistência/verificação em cima deles.

## Decisões de design

### Por que hash-chain simplificado (e não uma blockchain de verdade)

O objetivo do V1 é **detecção de adulteração local e auditabilidade determinística**, sem a complexidade operacional de uma blockchain real (rede P2P, consenso, mineração/validação distribuída, wallets, etc.). Para um sistema de governança de IA rodando localmente, o requisito é: "se alguém mexer num evento passado, eu quero saber". Um hash-chain single-writer, append-only, com verificação determinística resolve exatamente isso com uma fração da complexidade.

### Como funciona

- Cada evento tem um `hash = SHA256(prev_hash + event_type + actor + timestamp + json(payload, sort_keys=True))`.
- O `prev_hash` do evento N é o `hash` do evento N-1.
- O primeiro evento da cadeia usa um **hash gênese fixo**, documentado como `GENESIS_HASH = "0" * 64` (64 caracteres `"0"`, mesmo tamanho de um SHA-256 em hex) — convenção emprestada de blockchains reais (ex. o bloco gênese do Bitcoin também referencia um hash de zeros como "pai").
- Persistência em `.jsonl` (uma linha JSON por evento, append-only) — simples, legível por humano, versionável, e fácil de re-processar linha a linha.
- Serialização determinística do payload via `json.dumps(payload, sort_keys=True)`: garante que o mesmo conteúdo lógico sempre produza o mesmo hash, independentemente da ordem de inserção de chaves em memória (importante porque dicts Python não garantem ordem estável entre execuções diferentes, embora na prática desde 3.7 preservem ordem de inserção — `sort_keys=True` remove essa dependência por completo).
- O `timestamp` usado no cálculo do hash é o texto exato de `datetime.isoformat()` gravado no arquivo — e não a serialização JSON produzida pelo Pydantic (que usa sufixo `Z`) — para garantir que a string usada no hash seja *byte a byte* a mesma que é relida do disco. Essa foi uma armadilha real encontrada durante o desenvolvimento (ver seção de testes).

### Diferença para uma blockchain de verdade

| Aspecto | Este hash-chain (V1) | Blockchain real |
|---|---|---|
| Escritores | Único processo/arquivo local | Múltiplos nós distribuídos |
| Consenso | Nenhum (não é necessário — single writer) | Proof-of-Work / Proof-of-Stake / BFT, etc. |
| Replicação | Nenhuma — um único arquivo `.jsonl` | Réplicas em toda a rede |
| Resistência a adulteração | Detecta adulteração **se o atacante não recalcular a cadeia inteira** | Adulteração exigiria reescrever a cadeia em >50% dos nós simultaneamente |
| Imutabilidade | "Imutabilidade por convenção": nada impede reescrever o arquivo do zero | Imutabilidade prática por custo computacional/econômico distribuído |

### Limitações — sendo honesto sobre segurança

**Isto é auditoria de integridade local, não uma prova criptográfica contra um adversário com acesso total.**

- Um atacante (ou um bug) com acesso de **escrita** ao arquivo `audit_log.jsonl`, que também consiga rodar `verify_chain()` (ou simplesmente saiba como o hash é calculado), pode:
  1. Editar qualquer evento passado;
  2. Recalcular o `hash` desse evento e o `prev_hash`/`hash` de **todos** os eventos subsequentes;
  3. `verify_chain()` voltará a retornar `True` — a adulteração fica indetectável **por este mecanismo isolado**.
- Não há distribuição, não há consenso, não há ancoragem externa (ex. notarização em blockchain pública, timestamping de terceiros, WORM storage). O arquivo é tão confiável quanto o controle de acesso ao disco onde ele vive.
- O valor real deste mecanismo é: (a) detectar adulterações **acidentais** ou feitas por alguém que não conhece o esquema de hash; (b) fornecer uma trilha auditável e re-verificável determinística para compliance interno; (c) servir de base para uma evolução V2 com ancoragem externa real.
- **V2 (ROADMAP)** — "Blockchain Audit Layer": evolução planejada para um mecanismo real de ancoragem/distribuição (ex. replicar hashes para um destino write-once externo, ou blockchain pública/permissionada), que resolveria essa limitação.

In [1]:
import json
import sys
from pathlib import Path

# Garante que a raiz do repo está no sys.path (para importar core.* e shared.*).
# Convenção: notebook é executado a partir de notebooks/, raiz do repo é o pai.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from core.audit_logs.logger import AuditLogger, GENESIS_HASH
from shared.schemas import AuditEventType

print("REPO_ROOT:", REPO_ROOT)
print("GENESIS_HASH:", GENESIS_HASH)

REPO_ROOT: G:\Outros computadores\Meu computador\Controle Base\Projetos, Robos e Automação\Projetos Git\Projetos Extras (Portfolio)\LGPD e IA (Terminar)
GENESIS_HASH: 0000000000000000000000000000000000000000000000000000000000000000


## Demonstração prática

Vamos registrar 4 eventos de tipos diferentes (simulando chamadas reais de outros módulos), verificar que a cadeia é válida, e depois adulterar um evento no meio da cadeia para provar que `verify_chain()` detecta a violação.

In [2]:
import tempfile

demo_dir = Path(tempfile.mkdtemp(prefix="audit_logs_demo_"))
demo_log_path = demo_dir / "audit_log_demo.jsonl"
logger = AuditLogger(log_path=demo_log_path)

e1 = logger.record_event(
    AuditEventType.PII_SCAN,
    actor="pii_detection_module",
    payload={"findings_count": 2, "has_sensitive_data": True},
)
e2 = logger.record_event(
    AuditEventType.POLICY_EVALUATION,
    actor="policy_engine_module",
    payload={"policy_id": "POL-001", "status": "allow_with_mitigation"},
)
e3 = logger.record_event(
    AuditEventType.PROMPT_SECURITY_SCAN,
    actor="prompt_security_module",
    payload={"is_safe": True, "score": 0.94},
)
e4 = logger.record_event(
    AuditEventType.TRUST_SCORE_COMPUTED,
    actor="trust_score_module",
    payload={"score": 87.5, "risk_level": "low"},
)

for e in (e1, e2, e3, e4):
    print(f"{e.event_type.value:<28} actor={e.actor:<24} prev_hash={e.prev_hash[:12]}... hash={e.hash[:12]}...")

pii_scan                     actor=pii_detection_module     prev_hash=000000000000... hash=1806141cfa55...
policy_evaluation            actor=policy_engine_module     prev_hash=1806141cfa55... hash=6b2d7723809d...
prompt_security_scan         actor=prompt_security_module   prev_hash=6b2d7723809d... hash=feec84a1e3b7...
trust_score_computed         actor=trust_score_module       prev_hash=feec84a1e3b7... hash=1c6276803e0e...


In [3]:
print("Cadeia intacta -> verify_chain():", logger.verify_chain())
assert logger.verify_chain() is True

Cadeia intacta -> verify_chain(): True


In [4]:
# Agora adulteramos o payload do evento do meio (POLICY_EVALUATION) diretamente no arquivo,
# sem recalcular os hashes -- simulando um ataque/erro que edita o arquivo "na mão".
lines = demo_log_path.read_text(encoding="utf-8").strip().splitlines()
print(f"Total de linhas no log: {len(lines)}")

record = json.loads(lines[1])
print("Payload original:", record["payload"])
record["payload"] = {"policy_id": "POL-001", "status": "allow"}  # adulterado: mitigação removida
lines[1] = json.dumps(record)
demo_log_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

print("Payload adulterado:", json.loads(lines[1])["payload"])

Total de linhas no log: 4
Payload original: {'policy_id': 'POL-001', 'status': 'allow_with_mitigation'}
Payload adulterado: {'policy_id': 'POL-001', 'status': 'allow'}


In [5]:
print("Cadeia adulterada -> verify_chain():", logger.verify_chain())
assert logger.verify_chain() is False
print("\nComo esperado: a adulteração do evento do meio foi detectada.")

Cadeia adulterada -> verify_chain(): False

Como esperado: a adulteração do evento do meio foi detectada.


## Rodando a suíte de testes

In [6]:
import subprocess

# Usa o mesmo interpretador que está rodando este kernel (venv do projeto:
# C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe), sem hardcode de caminho.
python_exe = sys.executable
result = subprocess.run(
    [python_exe, "-m", "pytest", "core/audit_logs/tests", "-v"],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
)
print(result.stdout)
print(result.stderr)
print("Return code:", result.returncode)

============================= test session starts =============================
platform win32 -- Python 3.10.8, pytest-9.1.1, pluggy-1.6.0 -- C:\Users\Yuri_\.venvs\athenagov-ai\Scripts\python.exe
cachedir: .pytest_cache
rootdir: G:\Outros computadores\Meu computador\Controle Base\Projetos, Robos e Automação\Projetos Git\Projetos Extras (Portfolio)\LGPD e IA (Terminar)
plugins: anyio-4.14.2, cov-7.1.0
collecting ... collected 6 items

core/audit_logs/tests/test_logger.py::test_record_single_event_and_verify_chain PASSED [ 16%]
core/audit_logs/tests/test_logger.py::test_record_multiple_events_and_verify_chain PASSED [ 33%]
core/audit_logs/tests/test_logger.py::test_tampered_middle_event_breaks_chain PASSED [ 50%]
core/audit_logs/tests/test_logger.py::test_tampered_hash_also_breaks_chain PASSED [ 66%]
core/audit_logs/tests/test_logger.py::test_empty_or_missing_file_does_not_crash PASSED [ 83%]
core/audit_logs/tests/test_logger.py::test_first_event_uses_genesis_hash PASSED [100%]

=======

## Handoff Summary

### Capacidades entregues

- Log de auditoria imutável (hash-chain local, estilo blockchain simplificado) persistido em `.jsonl`, append-only.
- Registro de eventos tipados (`AuditEventType` de `shared/schemas.py`) com `event_id` (uuid4), timestamp UTC, hash SHA-256 encadeado ao evento anterior.
- Verificação determinística da cadeia inteira (`verify_chain()`), detectando adulteração de payload/actor/event_type/timestamp/hash ou quebra da sequência de `prev_hash`.
- Leitura auxiliar de todos os eventos (`read_events()`).
- Função de conveniência (`default_logger()`) para uso fácil por outros módulos, apontando para o log padrão do projeto.
- Tratamento robusto de arquivo vazio ou inexistente (não crasha, `verify_chain()` retorna `True` para cadeia vazia).
- Suíte de 6 testes pytest, todos verdes (ver saída real da execução acima), incluindo o teste crítico de adulteração de um evento no meio da cadeia.

### Assinatura pública exata

```python
# core/audit_logs/logger.py

GENESIS_HASH: str  # "0" * 64
DEFAULT_LOG_PATH: Path  # core/audit_logs/data/audit_log.jsonl

class AuditLogger:
    def __init__(self, log_path: str | Path = DEFAULT_LOG_PATH) -> None: ...

    def record_event(
        self,
        event_type: AuditEventType,
        actor: str,
        payload: dict[str, Any],
    ) -> AuditEvent: ...

    def verify_chain(self) -> bool: ...

    def read_events(self) -> list[AuditEvent]: ...

def default_logger() -> AuditLogger: ...
```

Uso típico por outro módulo:

```python
from core.audit_logs.logger import default_logger
from shared.schemas import AuditEventType

logger = default_logger()
event = logger.record_event(
    AuditEventType.PII_SCAN,
    actor="pii_detection_module",
    payload={"findings_count": 3, "has_sensitive_data": True},
)
```

### Limitações honestas

- **Não é uma blockchain distribuída**: single-writer, single-file, sem consenso, sem replicação, sem ancoragem externa.
- **Não é prova criptográfica contra um adversário com acesso de escrita total ao disco**: um atacante que edite um evento e recalcule toda a cadeia subsequente localmente faz `verify_chain()` voltar a passar. O mecanismo protege contra adulteração acidental ou por quem não conhece o esquema de hash, e fornece uma trilha auditável reproduzível — não uma garantia de imutabilidade absoluta.
- Não há controle de concorrência (locking) para múltiplos processos escrevendo simultaneamente no mesmo arquivo — fora de escopo do V1, assumindo um único processo orquestrador (Governance Copilot) como escritor.
- Não há rotação/particionamento de arquivo para logs muito grandes.

### O que fica para V2

- **Blockchain Audit Layer** (item explícito do ROADMAP V2): evolução do hash-chain atual para um mecanismo com ancoragem externa real e/ou distribuição — candidatos incluem replicação para storage WORM, timestamping notarial de terceiros, ou blockchain pública/permissionada — eliminando a limitação de "atacante com acesso de escrita pode recalcular a cadeia inteira".
- Locking/controle de concorrência para múltiplos escritores.
- Rotação de arquivo de log e/ou particionamento por período.
- Assinatura digital dos eventos (chave privada do processo emissor) como camada adicional de não-repúdio.